# Linha de base da divergência Esquerda × Direita — 52ª legislatura

Estima quanto dois espaços Word2Vec divergem **apenas por acaso**. Os discursos de Esquerda e Direita são misturados e sorteados em dois grupos com os mesmos tamanhos dos blocos reais (embaralhamento dos rótulos). Treina-se um Word2Vec em cada grupo, com os parâmetros da dissertação, alinha-se por Procrustes (até 15.000 âncoras) e mede-se a distância média de cosseno.

Também informa o tamanho do corpus de cada bloco.

**Dados de entrada:** corpus pré-processado em `legislatura/data/artigo1_preprocessado`, gerado por `scripts/01_preprocess_corpus.py` a partir dos dados da API da Câmara dos Deputados (ver `data/README_DATA.md`). O notebook procura essa pasta no Google Drive montado.

**Execução:** Google Colab → *Ambiente de execução → Executar tudo*. Tempo estimado: cerca de 1 hora.


In [ ]:
# 1) Conectar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Localizar a pasta pré-processada da dissertação
from pathlib import Path
RAIZ = Path('/content/drive/MyDrive')
candidatos = [RAIZ / 'legislatura' / 'data' / 'artigo1_preprocessado']
candidatos += list(RAIZ.glob('*/legislatura/data/artigo1_preprocessado'))
candidatos += list(RAIZ.glob('*/data/artigo1_preprocessado'))
PRE = next((c for c in candidatos if (c / 'corpus_artigo1_metadata.csv').exists()), None)
if PRE is None:
    raise SystemExit('Pasta artigo1_preprocessado não encontrada no Drive montado (ver Dados de entrada no topo).')
print('Pasta encontrada:', PRE)

SAIDA = RAIZ / 'MESTRADO' / 'resultados_linha_base_divergencia'
SAIDA.mkdir(parents=True, exist_ok=True)
PERCENTIL_R3 = 0.50   # percentil do limiar fixo da R3 (0.50 = mediana)

In [ ]:
# 3) Preparar o ambiente (instala o gensim se precisar)
import subprocess, sys
try:
    import gensim
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gensim>=4.4"], check=True)
try:
    from gensim.models import Word2Vec
except Exception as e:
    raise SystemExit("Vá em Ambiente de execução > Reiniciar sessão e depois Executar tudo de novo. (" + str(e) + ")")
import os, time, tempfile
import numpy as np
import pandas as pd
LEGS = [52, 53, 54, 55, 56, 57]
meta = pd.read_csv(PRE / "corpus_artigo1_metadata.csv")
if "bloco_ideologico" not in meta.columns:
    raise SystemExit("Coluna bloco_ideologico não encontrada. Colunas: " + ", ".join(meta.columns))
meta = meta[["id_legislatura", "bloco_ideologico"]]
print(meta["bloco_ideologico"].value_counts())
print("Núcleos de processamento:", os.cpu_count())

In [ ]:
# 4) Tamanho do corpus de cada bloco por legislatura
def ler_blocos(L):
    linhas = (PRE / "by_legislature" / f"leg{L}.txt").read_text(encoding="utf-8").splitlines()
    b = meta.loc[meta["id_legislatura"] == L, "bloco_ideologico"].values
    if len(b) != len(linhas):
        raise ValueError(f"Leg {L}: metadados={len(b)} texto={len(linhas)}")
    esq = [linhas[i] for i in np.where(b == "Left")[0]]
    dir_ = [linhas[i] for i in np.where(b == "Right")[0]]
    return esq, dir_

tam = []
for L in LEGS:
    esq, dir_ = ler_blocos(L)
    tam.append({"legislatura": L,
                "discursos_esquerda": len(esq),
                "tokens_esquerda_mi": round(sum(len(s.split()) for s in esq) / 1e6, 2),
                "discursos_direita": len(dir_),
                "tokens_direita_mi": round(sum(len(s.split()) for s in dir_) / 1e6, 2)})
    del esq, dir_
tamanho = pd.DataFrame(tam).set_index("legislatura")
tamanho.to_csv(SAIDA / "tamanho_corpus_blocos.csv", encoding="utf-8-sig")
tamanho

In [ ]:
# 5) Linha de base: embaralhamento dos rótulos de bloco
PARAMS = dict(vector_size=300, window=8, min_count=10, sg=1, hs=0, negative=10,
              sample=1e-4, epochs=10, seed=42, workers=os.cpu_count())
REAL = {52: 0.4922, 53: 0.4945, 54: 0.5075, 55: 0.5242, 56: 0.5280, 57: 0.5179}
TMP = Path(tempfile.mkdtemp())

def treinar(linhas, nome):
    arq = TMP / nome
    arq.write_text("\n".join(linhas) + "\n", encoding="utf-8")
    return Word2Vec(corpus_file=str(arq), **PARAMS).wv

def unit(m):
    n = np.linalg.norm(m, axis=1, keepdims=True); n[n == 0] = 1
    return m / n

def divergencia(kva, kvb, max_anc=15000):
    comum = set(kvb.key_to_index)
    anc = [t for t in kva.index_to_key if t in comum][:max_anc]
    X = unit(np.array([kva[t] for t in anc], dtype=np.float64))
    Y = unit(np.array([kvb[t] for t in anc], dtype=np.float64))
    U, _, Vt = np.linalg.svd(X.T @ Y)
    d = 1.0 - np.sum((X @ (U @ Vt)) * Y, axis=1)
    return float(d.mean()), float(np.median(d)), len(anc)

rng = np.random.default_rng(42)
res = []
for L in LEGS:
    t0 = time.time()
    esq, dir_ = ler_blocos(L)
    pool = esq + dir_
    idx = rng.permutation(len(pool))
    gA = [pool[i] for i in idx[:len(esq)]]
    gB = [pool[i] for i in idx[len(esq):]]
    del esq, dir_, pool
    kvA = treinar(gA, "a.txt"); kvB = treinar(gB, "b.txt")
    m, md, n = divergencia(kvA, kvB)
    res.append({"legislatura": L, "divergencia_real": REAL[L],
                "linha_base": round(m, 4), "linha_base_mediana": round(md, 4),
                "ancoras": n, "excesso_real_menos_base": round(REAL[L] - m, 4)})
    pd.DataFrame(res).to_csv(SAIDA / "linha_base_divergencia.csv", index=False, encoding="utf-8-sig")
    print(f"Leg {L}: linha de base = {m:.4f} | real = {REAL[L]:.4f} | {(time.time()-t0)/60:.1f} min")
print("Pronto.")

In [ ]:
# 6) Resultado final
base = pd.DataFrame(res).set_index("legislatura")
print("Arquivos salvos em:", SAIDA)
base